# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print the metadata
metadata = dataset.metadata  # metadata is an object
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, their fields, and entity IDs.
All Croissant entities are identified by their `@id` fields.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets

print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'Unnamed')}")

# For each record set, list fields/columns with their @id
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('fields', [])
    if not fields:
        print("  No fields found.")
    else:
        print("  Fields and Columns by @id:")
        for field in fields:
            print(f"    Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames using their `@id`.
We'll load all available record sets and display column names and sample records from the first record set.


In [ ]:
# Collect the @id of all record sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample records:")
    print(df.head())

# Pick the first record set for further analysis
if record_set_ids:
    first_record_set_id = record_set_ids[0]
else:
    first_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Categorizing/grouping data
When referencing fields/columns, use their `@id` as shown above.


In [ ]:
import numpy as np

# Analyze numeric fields from the first record set
if first_record_set_id is not None:
    df = dataframes[first_record_set_id]
    print(f"Columns for {first_record_set_id}: {df.columns.tolist()}")

    # Choose first numeric field by @id, fallback if not numeric
    numeric_field_id = None
    for col in df.columns:
        try:
            # Try to convert to numeric
            if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Filtering: threshold is mean value
        numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = numeric_series.mean()
        filtered_df = df[numeric_series > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        normalized_field = f"{numeric_field_id}_normalized"
        filtered_df[normalized_field] = (filtered_numeric - filtered_numeric.mean()) / filtered_numeric.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_field]].head())

        # Grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
We'll plot the distribution of the selected numeric field and the normalized values.

In [ ]:
import matplotlib.pyplot as plt

# Plot distributions if data available
if first_record_set_id is not None and numeric_field_id is not None and normalized_field in filtered_df.columns:
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    plt.hist(filtered_numeric.dropna(), bins=5)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')

    plt.subplot(1,2,2)
    plt.hist(filtered_df[normalized_field].dropna(), bins=5)
    plt.title(f"Normalized {numeric_field_id}")
    plt.xlabel(normalized_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and process a multi-table clinical/genetic dataset using Croissant and `mlcroissant`. We used `@id` references for all entities—record sets, fields, and columns—ensuring clarity and reproducibility.

Key takeaways:
- Dataset structure and metadata are accessible directly via `mlcroissant`.
- Record sets and fields can be referenced precisely by `@id`.
- Simple EDA and visualizations are possible even for small, highly structured scientific datasets.

Next steps can include more advanced filtering, cross-table joins, or exporting processed data for further research analyses.